In [1]:
import tensorflow as tf
import time

# Prefetch

In [2]:
class FileDataset(tf.data.Dataset):
  def read_files_in_batches(num_samples):
    # Opening the file
    time.sleep(0.03)

    for record in range(num_samples):
      # Reading each record from the file
      time.sleep(0.015)
      yield (record,)

  def __new__(cls, num_samples=3):
    return tf.data.Dataset.from_generator(
        cls.read_files_in_batches,
        output_signature=tf.TensorSpec(shape=(1,), dtype=tf.int64),
        args=(num_samples,)
    )

In [3]:
def benchmark(dataset, num_epochs=2):
  for epoch in range(num_epochs):
    for sample in dataset:
      # Perform training step
      time.sleep(0.01)

In [11]:
%%timeit
benchmark(FileDataset())

278 ms ± 12.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
%%timeit
benchmark(FileDataset().prefetch(1))

255 ms ± 1.75 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
%%timeit
benchmark(FileDataset().prefetch(tf.data.AUTOTUNE))

259 ms ± 3.42 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


# Cache

In [12]:
ds = tf.data.Dataset.range(5)
ds = ds.map(lambda x: x**2)
ds = ds.cache("cache.txt")
list(ds.as_numpy_iterator())

[np.int64(0), np.int64(1), np.int64(4), np.int64(9), np.int64(16)]

In [13]:
def mapped_function(s):
  # Do some preprocessing
  tf.py_function(lambda: time.sleep(0.03), [], ())
  return s

In [14]:
%%timeit -r1 -n1
benchmark(FileDataset().map(mapped_function), 5)

1.42 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [15]:
%%timeit -r1 -n1
benchmark(FileDataset().map(mapped_function).cache(), 5)

444 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
